In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
train = pd.read_csv('train.csv')


In [3]:
train[['Group', 'Number']] = train['PassengerId'].str.split('_', expand=True)
train[['Group', 'Number', 'Cabin']].head()

,Group,Number,Cabin
0,0001,01,B/0/P
1,0002,01,F/0/S
2,0003,01,A/0/S
3,0003,02,A/0/S
4,0004,01,F/1/S


In [4]:
train['Cabin'].fillna(value = 'Unknown/Unknown/Unknown', inplace=True)
train[['deck','num','side']] = train['Cabin'].str.split('/',expand = True)

In [5]:
train.drop(columns=['PassengerId','Cabin','Name'],inplace=True)

In [6]:
train[['VIP','CryoSleep']] = train[['VIP','CryoSleep']].fillna(value=0)
train['VIP'] = train['VIP'].map({False: 0, True: 1})
train['CryoSleep'] = train['CryoSleep'].map({False: 0, True: 1})
train['Transported'] = train['Transported'].map({False :0,True:1})

In [7]:
spend_features = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
for col in spend_features:
    train.loc[(train['CryoSleep'] == True) & (train[col].isnull()),col] = 0

In [8]:
train['Total_Money_temp'] = train[spend_features].sum(axis =1)
train.loc[(train['Total_Money_temp'] > 0)&(train['CryoSleep'].isnull()),'CryoSleep'] = False

C:\Users\PC\AppData\Local\Temp\ipykernel_14680\2457114216.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  train.loc[(train['Total_Money_temp'] > 0)&(train['CryoSleep'].isnull()),'CryoSleep'] = False


In [9]:
train['Total Spend'] = train[spend_features].sum(axis = 1)

In [10]:
from numpy import median
from sklearn.impute import SimpleImputer 
cols_to_impute = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
SI = SimpleImputer(missing_values = np.nan , strategy = 'mean')
train[cols_to_impute] = SI.fit_transform(train[cols_to_impute])
SI_MED = SimpleImputer(missing_values= np.nan , strategy=median)
train[['Age']] = SI_MED.fit_transform(train[['Age']])

In [11]:
train[['VIP','CryoSleep']] = train[['VIP','CryoSleep']].fillna(value=0)
train['VIP'] = train['VIP'].astype(int)
train['CryoSleep'] = train['CryoSleep'].astype(int)
train['Transported'] = train['Transported'].astype(int)

C:\Users\PC\AppData\Local\Temp\ipykernel_14680\2569228929.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train[['VIP','CryoSleep']] = train[['VIP','CryoSleep']].fillna(value=0)


In [12]:
from sklearn.preprocessing import StandardScaler
SS = StandardScaler()
train[cols_to_impute] = SS.fit_transform(train[cols_to_impute])

In [13]:
from sklearn.preprocessing import LabelEncoder
LE = LabelEncoder()
cols_to_encode = ['HomePlanet','Destination','deck','side']

for col in cols_to_encode:
    train[col] = LE.fit_transform(train[col])

In [14]:
cols_to_convert = ['Group','Number','num']
for col in cols_to_convert:
    train[col] = pd.to_numeric(train[col],errors = 'coerce')

In [15]:
num_median = train['num'].median()
train['num'].fillna(num_median, inplace=True)  

In [16]:
from sklearn.model_selection import train_test_split
features = [col for col in train.columns if col != 'Transported']
X = train[features]
y = train['Transported']
X_train,X_test , y_train , y_test = train_test_split(X,y, test_size = 0.2,random_state=42,stratify=y)

In [17]:
from sklearn.ensemble import RandomForestClassifier 
RFC = RandomForestClassifier(n_estimators=100,criterion="entropy")
RFC.fit(X_train,y_train)
y_pred = RFC.predict(X_test)

In [18]:
from sklearn.metrics import confusion_matrix,accuracy_score
cm = confusion_matrix(y_test,y_pred)
print(cm)
acc = accuracy_score(y_test,y_pred)
print(acc)

[[718 145]
 [210 666]]
0.7958596894767107


In [19]:
test_df = pd.read_csv('test.csv')

# Save PassengerId before anything
submission_ids = test_df['PassengerId'].copy()

# Apply same steps as test_df (split, drop, fill, encode, scale)
test_df[['Group', 'Number']] = test_df['PassengerId'].str.split('_', expand=True)

test_df['Cabin'].fillna(value='Unknown/Unknown/Unknown', inplace=True)
test_df[['deck', 'num', 'side']] = test_df['Cabin'].str.split('/', expand=True)

test_df.drop(columns=['PassengerId', 'Cabin', 'Name'], inplace=True)

test_df[['VIP', 'CryoSleep']] = test_df[['VIP', 'CryoSleep']].fillna(value=0)
test_df['VIP'] = test_df['VIP'].map({False: 0, True: 1})
test_df['CryoSleep'] = test_df['CryoSleep'].map({False: 0, True: 1})

spend_features = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_features:
    test_df.loc[(test_df['CryoSleep'] == True) & (test_df[col].isnull()), col] = 0

test_df['Total_Money_temp'] = test_df[spend_features].sum(axis=1)
test_df.loc[(test_df['Total_Money_temp'] > 0) & (test_df['CryoSleep'].isnull()), 'CryoSleep'] = False

test_df['Total Spend'] = test_df[spend_features].sum(axis=1)

from numpy import median
from sklearn.impute import SimpleImputer

cols_to_impute = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
SI = SimpleImputer(missing_values=np.nan, strategy='mean')
test_df[cols_to_impute] = SI.fit_transform(test_df[cols_to_impute])
SI_MED = SimpleImputer(missing_values=np.nan, strategy=median)
test_df[['Age']] = SI_MED.fit_transform(test_df[['Age']])

test_df[['VIP', 'CryoSleep']] = test_df[['VIP', 'CryoSleep']].fillna(value=0)
test_df['VIP'] = test_df['VIP'].astype(int)
test_df['CryoSleep'] = test_df['CryoSleep'].astype(int)

from sklearn.preprocessing import StandardScaler
SS = StandardScaler()
test_df[cols_to_impute] = SS.fit_transform(test_df[cols_to_impute])

from sklearn.preprocessing import LabelEncoder
LE = LabelEncoder()
cols_to_encode = ['HomePlanet', 'Destination', 'deck', 'side']

for col in cols_to_encode:
    test_df[col] = LE.fit_transform(test_df[col])

cols_to_convert = ['Group', 'Number', 'num']
for col in cols_to_convert:
    test_df[col] = pd.to_numeric(test_df[col], errors='coerce')

num_median = test_df['num'].median()
test_df['num'].fillna(num_median, inplace=True)

# Prepare test features with same columns as test_df
# (features was defined earlier as all columns except 'Transported')
test_df_processed = test_df[features]

# Then predict:
predictions = RFC.predict(test_df_processed)

# Build submission
submission = pd.DataFrame({
    'PassengerId': submission_ids,
    'Transported': predictions.astype(bool)
})
submission.to_csv('submission.csv', index=False)

C:\Users\PC\AppData\Local\Temp\ipykernel_14680\4112034622.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  test_df.loc[(test_df['Total_Money_temp'] > 0) & (test_df['CryoSleep'].isnull()), 'CryoSleep'] = False
C:\Users\PC\AppData\Local\Temp\ipykernel_14680\4112034622.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_df[['VIP', 'CryoSleep']] = test_df[['VIP', 'CryoSleep']].fillna(value=0)


In [20]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   HomePlanet        8693 non-null   int64  
 1   CryoSleep         8693 non-null   int64  
 2   Destination       8693 non-null   int64  
 3   Age               8693 non-null   float64
 4   VIP               8693 non-null   int64  
 5   RoomService       8693 non-null   float64
 6   FoodCourt         8693 non-null   float64
 7   ShoppingMall      8693 non-null   float64
 8   Spa               8693 non-null   float64
 9   VRDeck            8693 non-null   float64
 10  Transported       8693 non-null   int64  
 11  Group             8693 non-null   int64  
 12  Number            8693 non-null   int64  
 13  deck              8693 non-null   int64  
 14  num               8693 non-null   float64
 15  side              8693 non-null   int64  
 16  Total_Money_temp  8693 non-null   float64


In [21]:
train.to_csv('TEST1.csv',index = False)